# EcoHome Energy Advisor - RAG Setup

In this notebook, you'll set up the Retrieval-Augmented Generation (RAG) pipeline for the EcoHome Energy Advisor. This will allow the agent to access and cite relevant energy-saving tips and best practices.

## Learning Objectives
- Set up ChromaDB vector store
- Load and process energy-saving documents
- Create embeddings for document chunks
- Implement semantic search functionality
- Test the RAG pipeline

## Documents Available
All `.txt` files in `data/documents/` are loaded automatically, including:
- `tip_device_best_practices.txt` - Device-specific optimization tips
- `tip_energy_savings.txt` - General energy-saving strategies
- `tip_hvac_optimization.txt` - HVAC and heating optimization
- `tip_smart_home_automation.txt` - Smart home automation tips
- `tip_renewable_integration.txt` - Solar and renewable integration
- `tip_seasonal_management.txt` - Seasonal energy management
- `tip_energy_storage.txt` - Home battery and storage optimization


## 1. Import Required Libraries


In [39]:
# Import the necessary libraries for RAG setup
import os
import glob
from langchain_chroma  import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

In [40]:
load_dotenv()

True

## 2. Load and Process Documents


In [41]:
# Load ALL energy-saving tip documents from the documents folder.
# Using glob means any new .txt file dropped into data/documents/ is picked up
# automatically — no need to edit this list when expanding the knowledge base.

documents = []
document_paths = sorted(glob.glob("data/documents/*.txt"))

for doc_path in document_paths:
    if os.path.exists(doc_path):
        loader = TextLoader(doc_path)
        docs = loader.load()
        documents.extend(docs)
        print(f"Loaded {len(docs)} document(s) from {doc_path}")
    else:
        print(f"Warning: {doc_path} not found")

print(f"\nTotal documents loaded: {len(documents)}")


Loaded 1 document(s) from data/documents\tip_device_best_practices.txt
Loaded 1 document(s) from data/documents\tip_energy_savings.txt
Loaded 1 document(s) from data/documents\tip_energy_storage.txt
Loaded 1 document(s) from data/documents\tip_hvac_optimization.txt
Loaded 1 document(s) from data/documents\tip_renewable_integration.txt
Loaded 1 document(s) from data/documents\tip_seasonal_management.txt
Loaded 1 document(s) from data/documents\tip_smart_home_automation.txt

Total documents loaded: 7


## 3. Split Documents into Chunks


In [42]:
# Split documents into smaller chunks for better retrieval
# Use RecursiveCharacterTextSplitter with appropriate chunk_size and chunk_overlap
# Experiment with different chunk sizes (e.g., 500, 1000, 1500 characters)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

# Split the documents
splits = text_splitter.split_documents(documents)
print(f"Split {len(documents)} documents into {len(splits)} chunks")

# Show sample chunk
if splits:
    print(f"\nSample chunk (first 200 characters):")
    print(splits[0].page_content[:200] + "...")


Split 7 documents into 24 chunks

Sample chunk (first 200 characters):
Large devices like electric vehicles, washing machines and dishwashers often support delayed start or timer functions. Schedule these devices to run outside of peak electricity pricing hours or during...


## 4. Create Vector Store


In [43]:
import os, glob
print("Notebook is running from:", os.getcwd())
print("Files in data/documents/:", glob.glob("data/documents/*.txt"))
print("Does data/documents/ exist?:", os.path.isdir("data/documents"))

Notebook is running from: c:\Users\jonahs\Desktop\Langchain\EcoEnergy_Agent\ecohome_starter
Files in data/documents/: ['data/documents\\tip_device_best_practices.txt', 'data/documents\\tip_energy_savings.txt', 'data/documents\\tip_energy_storage.txt', 'data/documents\\tip_hvac_optimization.txt', 'data/documents\\tip_renewable_integration.txt', 'data/documents\\tip_seasonal_management.txt', 'data/documents\\tip_smart_home_automation.txt']
Does data/documents/ exist?: True


In [44]:
# Create a ChromaDB vector store
# Initialize OpenAIEmbeddings
# Create the vector store with the document chunks
# Persist the vector store to disk for future use

# Set up the persist directory
persist_directory = "data/vectorstore"
os.makedirs(persist_directory, exist_ok=True)

# Initialize embeddings
embeddings = OpenAIEmbeddings(
    api_key=os.getenv("OPENAI_API_KEY")
)
print("Number of splits:", len(splits))
print("First split preview:", splits[0].page_content[:100] if splits else "NO SPLITS")
print("Embeddings object:", embeddings)
# Create the vector store
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory=persist_directory
)

print(f"Vector store created and persisted to {persist_directory}")
print(f"Total vectors stored: {len(splits)}")

Number of splits: 24
First split preview: Large devices like electric vehicles, washing machines and dishwashers often support delayed start o
Embeddings object: client=<openai.resources.embeddings.Embeddings object at 0x000001B12376A150> async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000001B125817650> model='text-embedding-ada-002' dimensions=None deployment='text-embedding-ada-002' openai_api_version=None openai_api_base=None openai_api_type=None openai_proxy=None embedding_ctx_length=8191 openai_api_key=SecretStr('**********') openai_organization=None allowed_special=None disallowed_special=None chunk_size=1000 max_retries=2 request_timeout=None headers=None tiktoken_enabled=True tiktoken_model_name=None show_progress_bar=False model_kwargs={} skip_empty=False default_headers=None default_query=None retry_min_seconds=4 retry_max_seconds=20 http_client=None http_async_client=None check_embedding_ctx_length=True
Vector store created and persisted to data/vectors

## 5. Test the RAG Pipeline


In [45]:
# Test the search functionality
# Try different queries related to energy optimization
# Test queries like:
# - "electric vehicle charging tips"
# - "thermostat optimization"
# - "dishwasher energy saving"
# - "solar power maximization"

test_queries = [
    "electric vehicle charging tips",
    "thermostat optimization",
    "dishwasher energy saving",
    "solar power maximization",
    "HVAC system efficiency",
    "pool pump scheduling"
]

print("=== Testing Vector Search ===")
for query in test_queries:
    print(f"\nQuery: '{query}'")
    docs = vectorstore.similarity_search(query, k=2)
    for i, doc in enumerate(docs):
        print(f"  Result {i+1}: {doc.page_content[:100]}...")


=== Testing Vector Search ===

Query: 'electric vehicle charging tips'
  Result 1: Large devices like electric vehicles, washing machines and dishwashers often support delayed start o...
  Result 2: Large devices like electric vehicles, washing machines and dishwashers often support delayed start o...

Query: 'thermostat optimization'
  Result 1: Heating and cooling are usually the largest part of a home's energy use, so optimising your HVAC sys...
  Result 2: Heating and cooling are usually the largest part of a home's energy use, so optimising your HVAC sys...

Query: 'dishwasher energy saving'
  Result 1: Dishwasher Best Practices:
- Only run when completely full
- Use the energy-saving or eco mode when ...
  Result 2: Dishwasher Best Practices:
- Only run when completely full
- Use the energy-saving or eco mode when ...

Query: 'solar power maximization'
  Result 1: Integrating renewable energy, mainly rooftop solar, means using as much of your own generation as po...
  Result 2: I

## 6. Test the Search Tool


In [46]:
# Test the search_energy_tips tool from tools.py
# Import and test the tool with various queries
# Verify that it returns relevant results

from tools import search_energy_tips

# Test the search_energy_tips function
print("=== Testing search_energy_tips Tool ===")

test_queries = [
    "electric vehicle charging",
    "thermostat settings",
    "dishwasher optimization",
    "solar power tips"
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    result = search_energy_tips.invoke(
        input={
            "query": query, 
            "max_results": 3,
        }
    )
    
    if "error" in result:
        print(f"  Error: {result['error']}")
    else:
        print(f"  Found {result['total_results']} results")
        for i, tip in enumerate(result['tips']):
            print(f"    {i+1}. {tip['content'][:100]}...")
            print(f"       Source: {tip['source']}")
            print(f"       Relevance: {tip['relevance_score']}")


=== Testing search_energy_tips Tool ===

Query: 'electric vehicle charging'
  Found 3 results
    1. Large devices like electric vehicles, washing machines and dishwashers often support delayed start o...
       Source: data/documents\tip_device_best_practices.txt
       Relevance: high
    2. Large devices like electric vehicles, washing machines and dishwashers often support delayed start o...
       Source: data/documents\tip_device_best_practices.txt
       Relevance: high
    3. Large devices like electric vehicles, washing machines and dishwashers often support delayed start o...
       Source: data/documents\tip_device_best_practices.txt
       Relevance: medium

Query: 'thermostat settings'
  Found 3 results
    1. Heating and cooling are usually the largest part of a home's energy use, so optimising your HVAC sys...
       Source: data/documents\tip_hvac_optimization.txt
       Relevance: high
    2. Heating and cooling are usually the largest part of a home's energy use, so o